# Redes neurais recorrentes

No módulo anterior, abordámos representações semânticas ricas do texto. A arquitetura que temos usado captura o significado agregado das palavras numa frase, mas não tem em conta a **ordem** das palavras, porque a operação de agregação que segue as embeddings remove essa informação do texto original. Como estes modelos são incapazes de representar a ordenação das palavras, não conseguem resolver tarefas mais complexas ou ambíguas, como a geração de texto ou o question answering.

Para capturar o significado de uma sequência de texto, vamos usar uma arquitetura de rede neural chamada **rede neural recorrente**, ou RNN. Ao usar uma RNN, passamos a nossa frase pela rede um token de cada vez, e a rede produz um certo **estado**, que depois passamos novamente para a rede com o token seguinte.

![Imagem a mostrar um exemplo de geração com rede neural recorrente.](../../../../../translated_images/pt-PT/rnn.27f5c29c53d727b5.webp)

Dada a sequência de tokens de entrada $X_0,\dots,X_n$, a RNN cria uma sequência de blocos de rede neural, e treina essa sequência de ponta a ponta usando retropropagação. Cada bloco da rede recebe um par $(X_i,S_i)$ como entrada, e produz $S_{i+1}$ como resultado. O estado final $S_n$ ou saída $Y_n$ entra num classificador linear para produzir o resultado. Todos os blocos da rede partilham os mesmos pesos, e são treinados de ponta a ponta usando uma passagem de retropropagação.

> A figura acima mostra a rede neural recorrente na forma desenrolada (à esquerda), e na representação recorrente mais compacta (à direita). É importante perceber que todas as Células RNN têm os mesmos **pesos partilháveis**.

Como os vectores de estado $S_0,\dots,S_n$ são passados pela rede, a RNN é capaz de aprender dependências sequenciais entre palavras. Por exemplo, quando a palavra *not* aparece em algum ponto da sequência, pode aprender a negar certos elementos dentro do vector de estado.

No interior, cada célula RNN contém duas matrizes de pesos: $W_H$ e $W_I$, e o viés $b$. Em cada passo da RNN, dado o input $X_i$ e o estado de entrada $S_i$, o estado de saída é calculado como $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, onde $f$ é uma função de activação (frequentemente $\tanh$).

> Para problemas como geração de texto (que abordaremos na próxima unidade) ou tradução automática, também queremos obter algum valor de saída em cada passo da RNN. Neste caso, existe também outra matriz $W_O$, e a saída é calculada como $Y_i=f(W_O\times S_i+b_O)$.

Vamos ver como as redes neurais recorrentes podem ajudar-nos a classificar o nosso conjunto de notícias.

> Para o ambiente sandbox, precisamos correr a célula seguinte para garantir que a biblioteca necessária está instalada e os dados são pré-carregados. Se estiver a correr localmente, pode ignorar a célula seguinte.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Ao treinar modelos grandes, a alocação de memória da GPU pode tornar-se um problema. Também pode ser necessário experimentar diferentes tamanhos de minibatch, para que os dados caibam na memória da GPU, mas o treino seja suficientemente rápido. Se estiver a executar este código na sua própria máquina com GPU, pode experimentar ajustar o tamanho do minibatch para acelerar o treino.

> **Nota**: Certas versões dos drivers NVidia são conhecidas por não libertar a memória após o treino do modelo. Estamos a executar vários exemplos neste notebook, e isso pode fazer com que a memória esgote em certas configurações, especialmente se estiver a fazer os seus próprios experimentos na mesma notebook. Se encontrar erros estranhos ao começar a treinar o modelo, pode querer reiniciar o kernel do notebook.


In [3]:
batch_size = 16
embed_size = 64

## Classificador RNN simples

No caso de uma RNN simples, cada unidade recorrente é uma rede linear simples, que recebe um vetor de entrada e um vetor de estado, e produz um novo vetor de estado. Em Keras, isto pode ser representado pela camada `SimpleRNN`.

Embora possamos passar tokens codificados em one-hot diretamente para a camada RNN, isso não é uma boa ideia devido à sua alta dimensionalidade. Portanto, usaremos uma camada de embedding para diminuir a dimensionalidade dos vetores de palavras, seguida por uma camada RNN, e finalmente um classificador `Dense`.

> **Nota**: Em casos onde a dimensionalidade não é tão alta, por exemplo ao usar tokenização a nível de caracteres, pode fazer sentido passar tokens codificados em one-hot diretamente para a célula RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Nota:** Usamos aqui uma camada de embeddings não treinada para simplificar, mas para melhores resultados podemos usar uma camada de embeddings pré-treinada usando Word2Vec, como descrito na unidade anterior. Seria um bom exercício adaptar este código para trabalhar com embeddings pré-treinados.

Agora vamos treinar o nosso RNN. As RNNs em geral são bastante difíceis de treinar, porque uma vez que as células da RNN são desenroladas ao longo do comprimento da sequência, o número resultante de camadas envolvidas na retropropagação é bastante grande. Assim, precisamos selecionar uma taxa de aprendizagem menor e treinar a rede num conjunto de dados maior para obter bons resultados. Isto pode levar bastante tempo, por isso é preferível usar uma GPU.

Para acelerar as coisas, vamos apenas treinar o modelo RNN com títulos de notícias, omitindo a descrição. Pode tentar treinar com descrição e ver se consegue fazer o modelo treinar.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Nota** que a precisão é provavelmente mais baixa aqui, porque estamos a treinar apenas com títulos de notícias.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Agora que estamos a usar máscara, podemos treinar o modelo com todo o conjunto de dados de títulos e descrições.

> **Nota**: Já reparaste que temos usado um vetorizador treinado nos títulos das notícias, e não em todo o corpo do artigo? Potencialmente, isto pode causar que alguns dos tokens sejam ignorados, por isso é melhor voltar a treinar o vetorizador. No entanto, isso pode ter apenas um efeito muito pequeno, por isso vamos manter o vetorizador pré-treinado anterior para simplificar.


## LSTM: Memória de curto e longo prazo

Um dos principais problemas das RNNs é o **desvanecimento dos gradientes**. As RNNs podem ser bastante longas, e podem ter dificuldade em propagar os gradientes até à primeira camada da rede durante a retropropagação. Quando isto acontece, a rede não consegue aprender relações entre tokens distantes. Uma forma de evitar este problema é introduzir uma **gestão explícita do estado** usando **portas**. As duas arquiteturas mais comuns que introduzem portas são a **memória de curto e longo prazo** (LSTM) e a **unidade recorrente com portas** (GRU). Vamos abordar as LSTMs aqui.

![Imagem mostrando um exemplo de célula de memória de curto e longo prazo](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Uma rede LSTM é organizada de forma semelhante a uma RNN, mas existem dois estados que são passados de camada em camada: o estado real $c$, e o vetor oculto $h$. Em cada unidade, o vetor oculto $h_{t-1}$ é combinado com a entrada $x_t$, e juntos controlam o que acontece ao estado $c_t$ e à saída $h_t$ através de **portas**. Cada porta tem ativação sigmoide (resultado no intervalo $[0,1]$), que pode ser pensado como uma máscara bit a bit quando multiplicada pelo vetor estado. As LSTMs têm as seguintes portas (da esquerda para a direita na imagem acima):
* **porta de esquecimento** que determina quais os componentes do vetor $c_{t-1}$ que precisamos esquecer, e quais passar adiante.
* **porta de entrada** que determina quanta informação do vetor de entrada e do vetor oculto anterior deve ser incorporada no vetor de estado.
* **porta de saída** que toma o novo vetor de estado e decide quais dos seus componentes serão usados para produzir o novo vetor oculto $h_t$.

Os componentes do estado $c$ podem ser pensados como bandeiras que podem ser ativadas e desativadas. Por exemplo, quando encontramos o nome *Alice* na sequência, supomos que se refere a uma mulher, e erguemos a bandeira no estado que diz que temos um substantivo feminino na frase. Quando encontramos as palavras *e o Tom*, erguemos a bandeira que indica que temos um substantivo plural. Assim, manipulando o estado podemos acompanhar as propriedades gramaticais da frase.

> **Nota**: Aqui está um ótimo recurso para entender o funcionamento interno das LSTMs: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) por Christopher Olah.

Embora a estrutura interna de uma célula LSTM possa parecer complexa, o Keras esconde esta implementação dentro da camada `LSTM`, por isso a única coisa que precisamos fazer no exemplo acima é substituir a camada recorrente:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Nota** que treinar LSTMs também é bastante lento, e pode não notar muito aumento na precisão no início do treino. Pode precisar continuar a treinar durante algum tempo para alcançar uma boa precisão.


## Redes Neuronais Recorrentes bidirecionais e multicamadas

Nos nossos exemplos até agora, as redes recorrentes operam desde o início de uma sequência até ao fim. Isto parece-nos natural porque segue a mesma direção em que lemos ou ouvimos a fala. No entanto, para cenários que requerem acesso aleatório à sequência de entrada, faz mais sentido executar o cálculo recorrente em ambas as direções. Redes RNN que permitem cálculos em ambas as direções são chamadas redes **bidirecionais**, e podem ser criadas envolvendo a camada recorrente com uma camada especial `Bidirectonal`.

> **Nota**: A camada `Bidirectional` faz duas cópias da camada que contém, e define a propriedade `go_backwards` de uma dessas cópias para `True`, fazendo com que percorra a sequência na direção oposta.

Redes recorrentes, unidirecionais ou bidirecionais, capturam padrões dentro de uma sequência, e armazenam-nos em vetores de estado ou os devolvem como saída. Tal como nas redes convolucionais, podemos construir outra camada recorrente após a primeira para captar padrões de nível superior, construídos a partir de padrões de nível inferior extraídos pela primeira camada. Isto conduz à noção de uma **RNN multicamada**, que consiste em duas ou mais redes recorrentes, onde a saída da camada anterior é passada como entrada para a próxima camada.

![Imagem que mostra uma RNN de memória de longo curto multicamada](../../../../../translated_images/pt-PT/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Imagem retirada [deste maravilhoso post](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) por Fernando López.*

O Keras torna a construção destas redes uma tarefa fácil, porque basta adicionar mais camadas recorrentes ao modelo. Para todas as camadas, exceto a última, é necessário especificar o parâmetro `return_sequences=True`, porque precisamos que a camada devolva todos os estados intermédios, e não apenas o estado final do cálculo recorrente.

Vamos construir uma LSTM bidirecional de duas camadas para o nosso problema de classificação.

> **Nota** este código volta a demorar bastante tempo a completar, mas proporciona-nos a melhor precisão que vimos até agora. Portanto, talvez valha a pena esperar e ver o resultado.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs para outras tarefas

Até agora, focámo-nos em usar RNNs para classificar sequências de texto. Mas elas podem lidar com muitas mais tarefas, como geração de texto e tradução automática &mdash; consideraremos essas tarefas na próxima unidade.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
